# Tech Challenge · Fase 3 · Big Data & Analytics
## Pipeline do State of Data Brasil (2023, 2024 e 2025-26) na AWS

**Problema de negócio:** uma Instituição Financeira de grande porte quer expandir sua área de Dados, Analytics e IA e precisa entender o mercado brasileiro de profissionais de dados para definir contratação, capacitação e investimento em tecnologia.

**Fonte:** pesquisa State of Data Brasil (Data Hackers + Bain), as 3 edições mais recentes no Kaggle.

**Arquitetura (ambiente AWS Academy Lab, região us-east-1):**

```
Kaggle (3 CSV) → S3 Bronze → Glue Job 1 (Spark) → S3 Silver → Crawler
              → Glue Job 2 (Spark) → S3 Gold → Crawler → Glue Data Catalog → Athena (SQL) → gráficos
```

| Etapa | Serviço | Seção deste notebook |
|---|---|---|
| 1. Ingestão e organização | Amazon S3 | §1 |
| 2. ETL Bronze → Silver | AWS Glue Job + Spark | §2 |
| 3. Agregação Silver → Gold | AWS Glue Job + Spark | §3 |
| 4. Catalogação | Glue Crawlers + Data Catalog | §4 |
| 5. Consultas analíticas | Amazon Athena (SQL) | §5 |
| 6. Gráficos | PySpark + pandas + matplotlib | §6 |

> Os jobs das seções 2 e 3 foram executados como **Glue Jobs** (Script editor, Spark, role `LabRole`, 2 workers G.1X). O mesmo código roda neste notebook em um **Glue Interactive Session**.

### Configuração da sessão (Glue Interactive Session)
As *magics* abaixo só funcionam no Glue Notebook. Em um Jupyter comum, pule esta célula.

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%additional_python_modules matplotlib

In [ ]:
BUCKET = "techchallenge-fase3-statedata-geovana"
BRONZE = f"s3://{BUCKET}/bronze"
SILVER = f"s3://{BUCKET}/silver/survey"
GOLD   = f"s3://{BUCKET}/gold"

## 1. Ingestão e organização no S3 (camada Bronze)

Os 3 CSVs baixados do Kaggle foram renomeados (nomes simples, sem espaços) e enviados ao bucket, um por pasta de partição:

```
s3://techchallenge-fase3-statedata-geovana/
├── bronze/
│   ├── ano=2023/survey_2023.csv   (5.293 respostas · 399 colunas)
│   ├── ano=2024/survey_2024.csv   (5.217 respostas · 403 colunas)
│   └── ano=2025/survey_2025.csv   (3.495 respostas · 388 colunas)
├── silver/            ← saída do Glue Job 1 (Parquet)
├── gold/              ← saída do Glue Job 2 (Parquet)
├── scripts/           ← código dos Glue Jobs
└── athena-results/    ← resultados das consultas do Athena
```

A Bronze guarda o dado **exatamente como veio da fonte**, para permitir reprocessar tudo se uma regra de limpeza mudar.

O upload foi feito pelo console do S3. A célula abaixo faz o mesmo via código (boto3), como alternativa:

In [ ]:
import boto3
s3 = boto3.client("s3")
arquivos_locais = {2023: "survey_2023.csv", 2024: "survey_2024.csv", 2025: "survey_2025.csv"}
for ano, arq in arquivos_locais.items():
    s3.upload_file(arq, BUCKET, f"bronze/ano={ano}/{arq}")
    print("enviado:", f"s3://{BUCKET}/bronze/ano={ano}/{arq}")

## 2. ETL Bronze → Silver (Glue Job 1 · PySpark)

**Desafios de qualidade encontrados nos dados brutos e como foram tratados:**

| Problema | Tratamento |
|---|---|
| Nomes de colunas diferentes em cada ano (2023: `('P2_r ', '...')`, 2024/25: `2.r_...`) | Extração do **código da pergunta** de cada coluna e um **mapa** de colunas por ano |
| Mesma pergunta com letra diferente (modelo de trabalho é `2.r` em 2024 e `2.q` em 2025) | Mapeamento explícito no `MAPA` |
| Ordem das clouds diferente em 2023 (Azure primeiro) | Mapeamento explícito por nome |
| Booleanos como `True/False` (2024) e `0/1`/`1.0` (2023, 2025) | Padronização para inteiro 0/1 |
| Salário em faixas de texto, com **2 faixas digitadas errado** na fonte | Conversão para o **ponto médio** da faixa (`salario_estimado`) |
| Gestores não respondem “nível” | Nova coluna `senioridade` = nível ou “Gestor” |
| 3 respondentes duplicados | `dropDuplicates(ano, id)` |
| Respostas com quebra de linha e aspas dentro do CSV | Leitura com `multiLine` e `escape` |

**Resultado:** uma tabela única com 46 colunas padronizadas, em Parquet, particionada por `ano`.

In [ ]:
# =====================================================================
# GLUE JOB 1  -  BRONZE  ->  SILVER
# Tech Challenge Fase 3 - State of Data Brasil (2023, 2024, 2025)
#
# O que este script faz (a "batedeira" da cozinha):
#   1. Lê os 3 CSVs brutos da camada Bronze no S3
#   2. Descobre o "código da pergunta" de cada coluna (ex.: 1.b = gênero),
#      porque cada ano escreve o nome da coluna de um jeito diferente
#   3. Escolhe só as colunas que respondem às perguntas do projeto
#      e dá a elas um nome simples e igual nos 3 anos
#   4. Limpa e padroniza (tipos, salário numérico, senioridade, etc.)
#   5. Junta os 3 anos numa tabela só e grava em Parquet na Silver,
#      particionada por ano
# =====================================================================

import re
from functools import reduce
from pyspark.sql import SparkSession, functions as F

# ---------------------------------------------------------------------
# 0) CONFIGURAÇÃO  (troque o nome do bucket se for diferente)
# ---------------------------------------------------------------------
BUCKET = "techchallenge-fase3-statedata-geovana"
BRONZE = f"s3://{BUCKET}/bronze"
SILVER = f"s3://{BUCKET}/silver/survey"

ARQUIVOS = {
    2023: f"{BRONZE}/ano=2023/survey_2023.csv",
    2024: f"{BRONZE}/ano=2024/survey_2024.csv",
    2025: f"{BRONZE}/ano=2025/survey_2025.csv",
}

spark = SparkSession.builder.appName("bronze_to_silver").getOrCreate()


# ---------------------------------------------------------------------
# 1) DESCOBRIR O CÓDIGO DA PERGUNTA EM CADA COLUNA
#    2023:  "('P2_r ', 'Atualmente qual a sua forma de trabalho?')"  -> "2.r"
#    2024:  "2.r_modelo_de_trabalho_atual"                              -> "2.r"
#    2025:  "4.j.3 Uso e pago pelas soluções..."                        -> "4.j.3"
# ---------------------------------------------------------------------
def codigo_da_coluna(nome):
    if nome.startswith("('P0'"):
        return "0.a"                                   # id do respondente em 2023
    m = re.match(r"\('P(\d)_([a-z])(?:_(\d+))?", nome)  # formato 2023
    if m:
        return ".".join(p for p in m.groups() if p)
    m = re.match(r"(\d\.[a-z](?:\.\d+)?)", nome)        # formato 2024 / 2025
    return m.group(1) if m else None


# ---------------------------------------------------------------------
# 2) O "MAPA": nome novo  ->  código da pergunta em 2023, 2024, 2025
#    (é aqui que resolvemos as diferenças entre os anos)
# ---------------------------------------------------------------------
MAPA = [
    # nome_silver                 2023      2024      2025
    # --- perfil
    ("id_respondente",           "0.a",    "0.a",    "0.a"),
    ("idade",                    "1.a",    "1.a",    "1.a"),
    ("faixa_idade",              "1.a.1",  "1.a.1",  "1.a.1"),
    ("genero",                   "1.b",    "1.b",    "1.b"),
    ("cor_raca_etnia",           "1.c",    "1.c",    "1.c"),
    ("pcd",                      "1.d",    "1.d",    "1.d"),
    ("uf",                       "1.i.1",  "1.i.1",  "1.i.1"),
    ("regiao",                   "1.i.2",  "1.i.2",  "1.i.2"),
    ("nivel_ensino",             "1.l",    "1.l",    "1.l"),
    ("area_formacao",            "1.m",    "1.m",    "1.m"),
    # --- trabalho
    ("situacao_trabalho",        "2.a",    "2.a",    "2.a"),
    ("setor",                    "2.b",    "2.b",    "2.b"),
    ("porte_empresa",            "2.c",    "2.c",    "2.c"),
    ("gestor",                   "2.d",    "2.d",    "2.d"),
    ("cargo",                    "2.f",    "2.f",    "2.f"),
    ("nivel",                    "2.g",    "2.g",    "2.g"),
    ("faixa_salarial",           "2.h",    "2.h",    "2.h"),
    ("tempo_exp_dados",          "2.i",    "2.i",    "2.i"),
    ("satisfeito",               "2.k",    "2.k",    "2.k"),
    ("modelo_trabalho_atual",    "2.r",    "2.r",    "2.q"),   # muda em 2025!
    ("modelo_trabalho_ideal",    "2.s",    "2.s",    "2.r"),   # muda em 2025!
    ("area_atuacao",             "4.a.1",  "4.a.1",  "4.a.1"),
    # --- IA na empresa
    ("ia_prioridade_empresa",    "3.e",    "3.e",    "3.e"),
    # --- linguagens (1 = usa, 0 = não usa)
    ("ling_sql",                 "4.d.1",  "4.d.1",  "4.c.1"),
    ("ling_r",                   "4.d.2",  "4.d.2",  "4.c.2"),
    ("ling_python",              "4.d.3",  "4.d.3",  "4.c.3"),
    ("ling_scala",               "4.d.10", "4.d.10", "4.c.7"),
    # --- cloud (atenção: em 2023 a ordem é diferente!)
    ("cloud_aws",                "4.h.2",  "4.h.1",  "4.e.1"),
    ("cloud_gcp",                "4.h.3",  "4.h.2",  "4.e.2"),
    ("cloud_azure",              "4.h.1",  "4.h.3",  "4.e.3"),
    ("cloud_on_premise",         "4.h.6",  "4.h.6",  "4.e.6"),
    # --- ferramentas de BI
    ("bi_powerbi",               "4.j.1",  "4.j.1",  "4.g.1"),
    ("bi_tableau",               "4.j.3",  "4.j.3",  "4.g.3"),
    ("bi_looker",                "4.j.7",  "4.j.7",  "4.g.7"),
    ("bi_looker_studio",         "4.j.8",  "4.j.8",  "4.g.8"),
    ("bi_quicksight",            "4.j.9",  "4.j.9",  "4.g.9"),
    ("bi_so_planilhas",          "4.j.22", "4.j.17", "4.g.17"),
    # --- IA generativa no dia a dia
    ("ia_nao_usa",               "4.m.1",  "4.m.1",  "4.j.1"),
    ("ia_gratuita",              "4.m.2",  "4.m.2",  "4.j.2"),
    ("ia_paga_propria",          "4.m.3",  "4.m.3",  "4.j.3"),
    ("ia_paga_empresa",          "4.m.4",  "4.m.4",  "4.j.4"),
    ("ia_copilot_codigo",        "4.m.5",  "4.m.5",  "4.j.5"),
]
POSICAO_DO_ANO = {2023: 1, 2024: 2, 2025: 3}

COLUNAS_FLAG = [m[0] for m in MAPA if m[0].startswith(("ling_", "cloud_", "bi_", "ia_"))
                and m[0] != "ia_prioridade_empresa"] + ["gestor", "satisfeito"]


# ---------------------------------------------------------------------
# 3) LER CADA ANO E SELECIONAR AS COLUNAS DO MAPA
# ---------------------------------------------------------------------
def ler_ano(ano, caminho):
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)   # algumas respostas têm quebra de linha
          .option("escape", '"')       # aspas dentro de respostas
          .option("encoding", "UTF-8")
          .csv(caminho))

    # dicionário {código -> nome original da coluna}
    cod_para_coluna = {}
    for nome in df.columns:
        cod = codigo_da_coluna(nome)
        if cod and cod not in cod_para_coluna:
            cod_para_coluna[cod] = nome

    selecionadas = []
    for item in MAPA:
        nome_novo, cod = item[0], item[POSICAO_DO_ANO[ano]]
        original = cod_para_coluna.get(cod)
        if original is None:
            print(f"[AVISO] {ano}: código {cod} ({nome_novo}) não encontrado -> nulo")
            selecionadas.append(F.lit(None).cast("string").alias(nome_novo))
        else:
            # crases (`) porque os nomes têm pontos, parênteses e espaços
            selecionadas.append(F.col(f"`{original}`").cast("string").alias(nome_novo))

    return df.select(*selecionadas).withColumn("ano", F.lit(ano))


df = reduce(lambda a, b: a.unionByName(b),
            [ler_ano(ano, caminho) for ano, caminho in ARQUIVOS.items()])


# ---------------------------------------------------------------------
# 4) LIMPEZA E PADRONIZAÇÃO
# ---------------------------------------------------------------------
# 4.1 tira espaços e transforma texto vazio em nulo
for c in df.columns:
    if c != "ano":
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))

# 4.2 colunas sim/não viram 0 ou 1 (2024 vem como True/False, os outros como 0/1)
for c in COLUNAS_FLAG:
    v = F.lower(F.col(c))
    df = df.withColumn(c, F.when(v.isin("1", "1.0", "true"), 1)
                           .when(v.isin("0", "0.0", "false"), 0)
                           .otherwise(None).cast("int"))

# 4.3 idade como número
df = df.withColumn("idade", F.col("idade").cast("double").cast("int"))

# 4.4 salário: faixa (texto) -> valor do meio da faixa (número)
#     (inclui 2 faixas com erro de digitação que existem na base original)
SALARIO_MEIO = {
    "Menos de R$ 1.000/mês":             500,
    "de R$ 101/mês a R$ 2.000/mês":      1500,   # erro de digitação na base
    "de R$ 1.001/mês a R$ 2.000/mês":    1500,
    "de R$ 2.001/mês a R$ 3.000/mês":    2500,
    "de R$ 3.001/mês a R$ 4.000/mês":    3500,
    "de R$ 4.001/mês a R$ 6.000/mês":    5000,
    "de R$ 6.001/mês a R$ 8.000/mês":    7000,
    "de R$ 8.001/mês a R$ 12.000/mês":   10000,
    "de R$ 12.001/mês a R$ 16.000/mês":  14000,
    "de R$ 16.001/mês a R$ 20.000/mês":  18000,
    "de R$ 20.001/mês a R$ 25.000/mês":  22500,
    "de R$ 25.001/mês a R$ 30.000/mês":  27500,
    "de R$ 25.001/mês a R$ 3000/mês":    27500,  # erro de digitação na base
    "de R$ 30.001/mês a R$ 40.000/mês":  35000,
    "Acima de R$ 40.001/mês":            45000,
}
mapa_sal = F.create_map(*[x for k, v in SALARIO_MEIO.items() for x in (F.lit(k), F.lit(v))])
df = df.withColumn("salario_estimado", mapa_sal[F.col("faixa_salarial")].cast("int"))

# 4.5 senioridade: gestores não respondem "nível", então viram "Gestor"
df = df.withColumn(
    "senioridade",
    F.when(F.col("nivel").isNotNull(), F.col("nivel"))
     .when(F.col("gestor") == 1, F.lit("Gestor"))
)

# 4.6 modelo de trabalho com nomes curtos
def modelo_curto(col):
    return (F.when(col.contains("100% remoto"), "Remoto")
             .when(col.contains("híbrido flexível"), "Híbrido flexível")
             .when(col.contains("híbrido com dias fixos"), "Híbrido fixo")
             .when(col.contains("100% presencial"), "Presencial")
             .otherwise(col))
df = (df.withColumn("modelo_trabalho_atual", modelo_curto(F.lower(F.col("modelo_trabalho_atual"))))
        .withColumn("modelo_trabalho_ideal", modelo_curto(F.lower(F.col("modelo_trabalho_ideal")))))

# 4.7 indicador geral: a pessoa usa IA generativa no trabalho?
df = df.withColumn(
    "usa_ia_generativa",
    F.when((F.col("ia_gratuita") == 1) | (F.col("ia_paga_propria") == 1) |
           (F.col("ia_paga_empresa") == 1) | (F.col("ia_copilot_codigo") == 1), 1)
     .when(F.col("ia_nao_usa") == 1, 0)
)

# 4.8 remove linhas duplicadas, se houver
df = df.dropDuplicates(["ano", "id_respondente"])


# ---------------------------------------------------------------------
# 5) GRAVAR NA SILVER (Parquet, particionado por ano)
# ---------------------------------------------------------------------
(df.write
   .mode("overwrite")
   .partitionBy("ano")
   .parquet(SILVER))

print("Linhas por ano gravadas na Silver:")
df.groupBy("ano").count().orderBy("ano").show()


**Saída do job (validada no Athena):** 2023 = 5.293 · 2024 = 5.215 · 2025 = 3.494 linhas.

## 3. Agregação Silver → Gold (Glue Job 2 · PySpark)

A Gold transforma a tabela de respondentes em **9 tabelas pequenas, uma por pergunta de negócio**. Técnica central: em colunas 0/1, a **média é a porcentagem** de quem marcou a opção.

In [ ]:
# =====================================================================
# GLUE JOB 2  -  SILVER  ->  GOLD
# Tech Challenge Fase 3 - State of Data Brasil (2023, 2024, 2025)
#
# A Silver tem 1 linha por pessoa. A Gold tem tabelas pequenas e já
# resumidas, uma para cada pergunta de negócio. Cada tabela Gold vira
# um (ou mais) gráfico(s) na apresentação.
#
#   gold_kpis               -> números gerais por ano (capa / resumo)
#   gold_cargos             -> como o mercado está estruturado
#   gold_senioridade        -> perfis mais valorizados (salário)
#   gold_genero             -> diversidade de gênero e diferença salarial
#   gold_tecnologias        -> tecnologias mais adotadas
#   gold_ia_uso             -> adoção de IA generativa pelos profissionais
#   gold_ia_empresas        -> IA como prioridade nas empresas
#   gold_regiao             -> diferenças entre regiões
#   gold_modelo_trabalho    -> remoto x híbrido x presencial
# =====================================================================

from pyspark.sql import SparkSession, functions as F, Window

BUCKET = "techchallenge-fase3-statedata-geovana"
SILVER = f"s3://{BUCKET}/silver/survey"
GOLD = f"s3://{BUCKET}/gold"

spark = SparkSession.builder.appName("silver_to_gold").getOrCreate()
df = spark.read.parquet(SILVER)

# Funções de apoio -----------------------------------------------------
def pct(col):
    """Média de uma coluna 0/1 = porcentagem de quem marcou."""
    return F.round(F.avg(col) * 100, 1)

def sal_medio():
    return F.round(F.avg("salario_estimado"), 0)

def sal_mediano():
    return F.percentile_approx("salario_estimado", 0.5)

def pct_do_total(particao="ano"):
    """% que cada linha representa dentro do ano."""
    w = Window.partitionBy(particao)
    return F.round(F.col("qtd") / F.sum("qtd").over(w) * 100, 1)

def salvar(tabela, nome):
    tabela.write.mode("overwrite").parquet(f"{GOLD}/{nome}")
    print(f"[OK] {nome}: {tabela.count()} linhas")

empregados = df.filter(F.col("salario_estimado").isNotNull())


# 1) KPIs gerais por ano ----------------------------------------------
kpis = df.groupBy("ano").agg(
    F.count("*").alias("respondentes"),
    F.round(F.avg("idade"), 1).alias("idade_media"),
    pct((F.col("genero") == "Feminino").cast("int")).alias("pct_mulheres"),
    sal_medio().alias("salario_medio"),
    sal_mediano().alias("salario_mediano"),
    pct("usa_ia_generativa").alias("pct_usa_ia"),
    pct((F.col("modelo_trabalho_atual") == "Remoto").cast("int")).alias("pct_remoto"),
    pct("ling_python").alias("pct_python"),
    pct("cloud_aws").alias("pct_aws"),
)
salvar(kpis, "gold_kpis")


# 2) Estrutura do mercado: cargos -------------------------------------
cargos = (empregados.filter(F.col("cargo").isNotNull())
          .groupBy("ano", "cargo")
          .agg(F.count("*").alias("qtd"),
               sal_medio().alias("salario_medio"),
               sal_mediano().alias("salario_mediano"))
          .withColumn("pct_do_ano", pct_do_total()))
salvar(cargos, "gold_cargos")


# 3) Perfis mais valorizados: senioridade -----------------------------
senioridade = (empregados.filter(F.col("senioridade").isNotNull())
               .groupBy("ano", "senioridade")
               .agg(F.count("*").alias("qtd"),
                    sal_medio().alias("salario_medio"),
                    sal_mediano().alias("salario_mediano"),
                    pct("usa_ia_generativa").alias("pct_usa_ia"))
               .withColumn("pct_do_ano", pct_do_total())
               .withColumn("ordem", F.when(F.col("senioridade") == "Júnior", 1)
                                     .when(F.col("senioridade") == "Pleno", 2)
                                     .when(F.col("senioridade") == "Sênior", 3)
                                     .when(F.col("senioridade") == "Especialista/Staff+", 4)
                                     .when(F.col("senioridade") == "Gestor", 5)))
salvar(senioridade, "gold_senioridade")


# 4) Diversidade de gênero --------------------------------------------
genero = (df.filter(F.col("genero").isin("Masculino", "Feminino"))
          .groupBy("ano", "genero", "senioridade")
          .agg(F.count("*").alias("qtd"),
               sal_medio().alias("salario_medio")))
salvar(genero, "gold_genero")


# 5) Tecnologias: formato "longo" (uma linha por tecnologia) ----------
TECNOLOGIAS = [
    ("Linguagem", "SQL", "ling_sql"), ("Linguagem", "Python", "ling_python"),
    ("Linguagem", "R", "ling_r"), ("Linguagem", "Scala", "ling_scala"),
    ("Cloud", "AWS", "cloud_aws"), ("Cloud", "Azure", "cloud_azure"),
    ("Cloud", "GCP", "cloud_gcp"), ("Cloud", "On premise", "cloud_on_premise"),
    ("BI", "Power BI", "bi_powerbi"), ("BI", "Tableau", "bi_tableau"),
    ("BI", "Looker", "bi_looker"), ("BI", "Looker Studio", "bi_looker_studio"),
    ("BI", "QuickSight", "bi_quicksight"), ("BI", "Só planilhas", "bi_so_planilhas"),
]
tecnologias = None
for categoria, nome, coluna in TECNOLOGIAS:
    parte = (df.groupBy("ano")
               .agg(pct(coluna).alias("pct_uso"),
                    F.count(coluna).alias("respondentes"))
               .withColumn("categoria", F.lit(categoria))
               .withColumn("tecnologia", F.lit(nome)))
    tecnologias = parte if tecnologias is None else tecnologias.unionByName(parte)
salvar(tecnologias.select("ano", "categoria", "tecnologia", "pct_uso", "respondentes"),
       "gold_tecnologias")


# 6) IA generativa: uso pelos profissionais ---------------------------
ia_uso = (df.filter(F.col("usa_ia_generativa").isNotNull())
          .groupBy("ano")
          .agg(F.count("*").alias("respondentes"),
               pct("usa_ia_generativa").alias("pct_usa_ia"),
               pct("ia_nao_usa").alias("pct_nao_usa"),
               pct("ia_gratuita").alias("pct_so_gratuita"),
               pct("ia_paga_propria").alias("pct_paga_do_bolso"),
               pct("ia_paga_empresa").alias("pct_empresa_paga"),
               pct("ia_copilot_codigo").alias("pct_ia_para_codigo")))
salvar(ia_uso, "gold_ia_uso")


# 7) IA generativa: prioridade nas empresas (pergunta feita a gestores)
ia_empresas = (df.filter(F.col("ia_prioridade_empresa").isNotNull())
               .withColumn("prioridade", F.when(F.col("ia_prioridade_empresa").startswith("Sim, é nossa principal"), "Principal prioridade")
                                          .when(F.col("ia_prioridade_empresa").startswith("Sim, está entre"), "Entre as prioridades")
                                          .when(F.col("ia_prioridade_empresa").startswith("Mais ou menos"), "Iniciativas isoladas")
                                          .when(F.col("ia_prioridade_empresa").startswith("Não é"), "Não é prioridade")
                                          .otherwise("Não sabe"))
               .groupBy("ano", "prioridade").agg(F.count("*").alias("qtd"))
               .withColumn("pct_do_ano", pct_do_total()))
salvar(ia_empresas, "gold_ia_empresas")


# 8) Regiões ----------------------------------------------------------
regiao = (df.filter(F.col("regiao").isNotNull())
          .groupBy("ano", "regiao")
          .agg(F.count("*").alias("qtd"),
               sal_medio().alias("salario_medio"),
               pct((F.col("modelo_trabalho_atual") == "Remoto").cast("int")).alias("pct_remoto"),
               pct((F.col("genero") == "Feminino").cast("int")).alias("pct_mulheres"))
          .withColumn("pct_do_ano", pct_do_total()))
salvar(regiao, "gold_regiao")


# 9) Modelo de trabalho: atual x ideal --------------------------------
atual = (df.filter(F.col("modelo_trabalho_atual").isNotNull())
         .groupBy("ano", F.col("modelo_trabalho_atual").alias("modelo"))
         .agg(F.count("*").alias("qtd"), sal_medio().alias("salario_medio"))
         .withColumn("pct_do_ano", pct_do_total())
         .withColumn("tipo", F.lit("Atual")))
ideal = (df.filter(F.col("modelo_trabalho_ideal").isNotNull())
         .groupBy("ano", F.col("modelo_trabalho_ideal").alias("modelo"))
         .agg(F.count("*").alias("qtd"), F.lit(None).cast("double").alias("salario_medio"))
         .withColumn("pct_do_ano", pct_do_total())
         .withColumn("tipo", F.lit("Ideal")))
salvar(atual.unionByName(ideal), "gold_modelo_trabalho")

print("Camada Gold concluída!")


## 4. Catalogação (Glue Crawlers + Data Catalog)

- Database `statedata_db`
- `crawler_silver` → `s3://.../silver/survey/` com prefixo `silver_` → tabela **`silver_survey`** (partição `ano` detectada automaticamente)
- `crawler_gold` → `s3://.../gold/` → tabelas **`gold_kpis`, `gold_cargos`, `gold_senioridade`, `gold_genero`, `gold_tecnologias`, `gold_ia_uso`, `gold_ia_empresas`, `gold_regiao`, `gold_modelo_trabalho`**

Os crawlers foram criados pelo console. Equivalente via código:

In [ ]:
glue = boto3.client("glue")
ROLE = "LabRole"
glue.create_database(DatabaseInput={"Name": "statedata_db"})
glue.create_crawler(Name="crawler_silver", Role=ROLE, DatabaseName="statedata_db", TablePrefix="silver_",
                    Targets={"S3Targets": [{"Path": f"s3://{BUCKET}/silver/survey/"}]})
glue.create_crawler(Name="crawler_gold", Role=ROLE, DatabaseName="statedata_db",
                    Targets={"S3Targets": [{"Path": f"s3://{BUCKET}/gold/"}]})
for c in ["crawler_silver", "crawler_gold"]:
    glue.start_crawler(Name=c)

## 5. Consultas analíticas (Amazon Athena · SQL)

As consultas abaixo foram executadas no editor do Athena (database `statedata_db`). Estão organizadas pelas perguntas do desafio. O arquivo `consultas_athena.sql` traz as mesmas consultas para copiar e colar.

As células usam a magic `%%sql` do Glue Notebook (que executa via Spark SQL sobre o Data Catalog).

In [ ]:
%%sql
USE statedata_db

In [ ]:
%%sql


-- ---------------------------------------------------------------------
-- 0. VALIDAÇÃO DO PIPELINE: quantas respostas por ano chegaram na Silver
-- ---------------------------------------------------------------------
SELECT ano, COUNT(*) AS respondentes
FROM silver_survey
GROUP BY ano
ORDER BY ano;

In [ ]:
%%sql
-- ---------------------------------------------------------------------
-- 0.1 VISÃO GERAL (KPIs por ano)
-- ---------------------------------------------------------------------
SELECT *
FROM gold_kpis
ORDER BY ano;

In [ ]:
%%sql
-- =====================================================================
-- P1. COMO ESTÁ ESTRUTURADO O MERCADO BRASILEIRO DE DADOS?
-- =====================================================================

-- 1.1 Cargos mais comuns em 2025 (participação e salário)
SELECT cargo, qtd, pct_do_ano, salario_medio
FROM gold_cargos
WHERE ano = 2025
ORDER BY qtd DESC
LIMIT 10;

In [ ]:
%%sql
-- 1.2 Distribuição por senioridade ao longo dos anos
SELECT ano, senioridade, qtd, pct_do_ano
FROM gold_senioridade
ORDER BY ano, ordem;

In [ ]:
%%sql
-- 1.3 Setores que mais empregam profissionais de dados (2025)
SELECT setor,
       COUNT(*) AS qtd,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM silver_survey
WHERE ano = 2025 AND setor IS NOT NULL
GROUP BY setor
ORDER BY qtd DESC
LIMIT 10;

In [ ]:
%%sql
-- 1.4 Escolaridade dos profissionais (2025)
SELECT nivel_ensino,
       COUNT(*) AS qtd,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM silver_survey
WHERE ano = 2025 AND nivel_ensino IS NOT NULL
GROUP BY nivel_ensino
ORDER BY qtd DESC;

In [ ]:
%%sql
-- =====================================================================
-- P2. QUAIS PERFIS PROFISSIONAIS SÃO MAIS VALORIZADOS?
-- =====================================================================

-- 2.1 Salário médio por senioridade e ano
SELECT ano, senioridade, salario_medio, salario_mediano
FROM gold_senioridade
ORDER BY ano, ordem;

In [ ]:
%%sql
-- 2.2 Cargos com maior salário médio em 2025 (mínimo de 30 respondentes)
SELECT cargo, qtd, salario_medio
FROM gold_cargos
WHERE ano = 2025 AND qtd >= 30
ORDER BY salario_medio DESC
LIMIT 10;

In [ ]:
%%sql
-- 2.3 Evolução do salário médio geral (crescimento % ano a ano)
SELECT ano,
       salario_medio,
       ROUND((salario_medio - LAG(salario_medio) OVER (ORDER BY ano))
             * 100.0 / LAG(salario_medio) OVER (ORDER BY ano), 1) AS cresc_pct
FROM gold_kpis
ORDER BY ano;

In [ ]:
%%sql
-- =====================================================================
-- P3. QUAL É O CENÁRIO DE DIVERSIDADE DE GÊNERO?
-- =====================================================================

-- 3.1 Participação feminina por ano
SELECT ano, pct_mulheres
FROM gold_kpis
ORDER BY ano;

In [ ]:
%%sql
-- 3.2 Diferença salarial entre homens e mulheres por senioridade
SELECT ano,
       senioridade,
       MAX(CASE WHEN genero = 'Masculino' THEN salario_medio END) AS sal_homens,
       MAX(CASE WHEN genero = 'Feminino'  THEN salario_medio END) AS sal_mulheres,
       ROUND((MAX(CASE WHEN genero = 'Masculino' THEN salario_medio END)
            - MAX(CASE WHEN genero = 'Feminino'  THEN salario_medio END)) * 100.0
            / MAX(CASE WHEN genero = 'Masculino' THEN salario_medio END), 1) AS gap_pct
FROM gold_genero
WHERE senioridade IS NOT NULL
GROUP BY ano, senioridade
ORDER BY ano, senioridade;

In [ ]:
%%sql
-- 3.3 Participação feminina por nível (2025): elas chegam aos cargos altos?
SELECT senioridade,
       SUM(qtd) AS total,
       ROUND(SUM(CASE WHEN genero = 'Feminino' THEN qtd ELSE 0 END) * 100.0 / SUM(qtd), 1) AS pct_mulheres
FROM gold_genero
WHERE ano = 2025 AND senioridade IS NOT NULL
GROUP BY senioridade
ORDER BY pct_mulheres DESC;

In [ ]:
%%sql
-- =====================================================================
-- P4. QUAIS TECNOLOGIAS TÊM MAIOR ADOÇÃO?
-- =====================================================================

-- 4.1 Ranking de tecnologias em 2025 por categoria
SELECT categoria, tecnologia, pct_uso
FROM gold_tecnologias
WHERE ano = 2025
ORDER BY categoria, pct_uso DESC;

In [ ]:
%%sql
-- 4.2 Evolução das clouds (2023 -> 2025)
SELECT tecnologia,
       MAX(CASE WHEN ano = 2023 THEN pct_uso END) AS pct_2023,
       MAX(CASE WHEN ano = 2024 THEN pct_uso END) AS pct_2024,
       MAX(CASE WHEN ano = 2025 THEN pct_uso END) AS pct_2025
FROM gold_tecnologias
WHERE categoria = 'Cloud'
GROUP BY tecnologia
ORDER BY pct_2025 DESC;

In [ ]:
%%sql
-- =====================================================================
-- P5. ADOÇÃO DE INTELIGÊNCIA ARTIFICIAL E SEU IMPACTO
-- =====================================================================

-- 5.1 Uso de IA generativa pelos profissionais
SELECT *
FROM gold_ia_uso
ORDER BY ano;

In [ ]:
%%sql
-- 5.2 IA como prioridade nas empresas (resposta de gestores)
SELECT ano, prioridade, qtd, pct_do_ano
FROM gold_ia_empresas
ORDER BY ano, pct_do_ano DESC;

In [ ]:
%%sql
-- 5.3 Impacto: quem tem a IA paga pela empresa ganha mais? (2025)
SELECT CASE WHEN ia_paga_empresa = 1 THEN 'Empresa paga a IA'
            ELSE 'Empresa não paga' END AS grupo,
       COUNT(*) AS qtd,
       ROUND(AVG(salario_estimado), 0) AS salario_medio
FROM silver_survey
WHERE ano = 2025 AND usa_ia_generativa IS NOT NULL AND salario_estimado IS NOT NULL
GROUP BY 1;

In [ ]:
%%sql
-- =====================================================================
-- P6. DIFERENÇAS ENTRE REGIÕES, SENIORIDADES E MODELOS DE TRABALHO
-- =====================================================================

-- 6.1 Regiões: concentração, salário e remoto
SELECT ano, regiao, qtd, pct_do_ano, salario_medio, pct_remoto
FROM gold_regiao
ORDER BY ano, qtd DESC;

In [ ]:
%%sql
-- 6.2 Modelo de trabalho atual x ideal (2025)
SELECT modelo,
       MAX(CASE WHEN tipo = 'Atual' THEN pct_do_ano END) AS pct_atual,
       MAX(CASE WHEN tipo = 'Ideal' THEN pct_do_ano END) AS pct_ideal,
       MAX(CASE WHEN tipo = 'Atual' THEN salario_medio END) AS salario_medio
FROM gold_modelo_trabalho
WHERE ano = 2025
GROUP BY modelo
ORDER BY pct_atual DESC;

In [ ]:
%%sql
-- 6.3 Evolução do trabalho 100% remoto
SELECT ano, pct_remoto
FROM gold_kpis
ORDER BY ano;

In [ ]:
%%sql
-- =====================================================================
-- P7. OPORTUNIDADES E DESAFIOS (consultas de apoio às recomendações)
-- =====================================================================

-- 7.1 Satisfação por modelo de trabalho (risco de retenção) · 2025
SELECT modelo_trabalho_atual,
       COUNT(*) AS qtd,
       ROUND(AVG(satisfeito) * 100, 1) AS pct_satisfeitos
FROM silver_survey
WHERE ano = 2025 AND modelo_trabalho_atual IS NOT NULL AND satisfeito IS NOT NULL
GROUP BY modelo_trabalho_atual
ORDER BY pct_satisfeitos DESC;

In [ ]:
%%sql
-- 7.2 Perfis escassos e caros: cargos com pouca oferta e salário alto (2025)
SELECT cargo, qtd, pct_do_ano, salario_medio
FROM gold_cargos
WHERE ano = 2025 AND qtd >= 20 AND pct_do_ano < 6
ORDER BY salario_medio DESC
LIMIT 5;

**Principais resultados das consultas**

| Pergunta | Achado (2025, salvo indicação) |
|---|---|
| Estrutura do mercado | Analista (24%), Cientista (17%) e Engenheiro de Dados (16%) somam 57% dos cargos · **Finanças/Bancos é o setor nº 1 (18,5%)** · 57% têm pós, mestrado ou doutorado |
| Perfis valorizados | Júnior R\$ 4,2 mil → Pleno 8,3 → Sênior 14,5 → Especialista 20,0 → Gestor 20,2 · ML/AI Engineer lidera (R\$ 19,1 mil) com só 4% do mercado |
| Diversidade | Mulheres 24% → 23,5% → 22% · caem de 28,5% (Júnior) para 18,4% (Gestor) · gap salarial de 19% no Sênior |
| Tecnologias | Python 92%, SQL 84%, Power BI 59% · AWS passou a Azure: 31% → 48% |
| IA | Uso 81% → 98% · empresa paga a IA: 6% → 42% · IA é prioridade em 61% das empresas (36% em 2023) |
| Regiões / modelos | Sudeste concentra 64% · remoto caiu de 46% para 40% · só 2% querem presencial e satisfação no presencial é 54% (vs 75% no híbrido flexível) |

## 6. Geração dos gráficos

As tabelas Gold são lidas com Spark, convertidas para pandas (são pequenas) e plotadas com matplotlib. Os PNGs gerados entram na apresentação executiva.

In [ ]:
"""
Geração dos gráficos da apresentação a partir das tabelas GOLD.

Uso:
    from graficos import gerar_todos
    gerar_todos(tabelas, pasta_saida="graficos")
onde `tabelas` é um dicionário {"gold_kpis": DataFrame, ...} (pandas).
"""
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# ---- Paleta (validada para daltonismo nas 3 primeiras cores) ----------
AZUL, LARANJA, VERDE_AGUA = "#2a78d6", "#eb6834", "#1baf7a"
CINZA = "#c9c8c2"          # categorias de contexto (não destacadas)
TXT, TXT2, GRADE = "#0b0b0b", "#52514e", "#e6e5e0"
COR_ANO = {2023: CINZA, 2024: "#86b6ef", 2025: AZUL}   # anos = rampa de um tom só

plt.rcParams.update({
    "font.family": "Inter", "font.size": 15, "text.color": TXT,
    "axes.edgecolor": GRADE, "axes.labelcolor": TXT2, "xtick.color": TXT2, "ytick.color": TXT2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": False, "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.dpi": 200, "legend.frameon": False,
})

def _brl(v):
    return f"R$ {v/1000:.1f} mil".replace(".", ",")

def _pct(v):
    return f"{v:.0f}%" if abs(v) >= 10 else f"{v:.1f}%".replace(".", ",")

def _salvar(fig, pasta, nome):
    fig.tight_layout()
    caminho = os.path.join(pasta, nome)
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)
    return caminho

def _barras_h(ax, rotulos, valores, cores, fmt, maxv=None):
    """Barras horizontais com valor escrito na ponta (ranking)."""
    y = range(len(rotulos))[::-1]
    ax.barh(list(y), valores, color=cores, height=0.62)
    maxv = maxv or max(valores)
    for yi, v in zip(y, valores):
        ax.text(v + maxv * 0.012, yi, fmt(v), va="center", fontsize=14, color=TXT)
    ax.set_yticks(list(y)); ax.set_yticklabels(rotulos, fontsize=14, color=TXT)
    ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    ax.set_xlim(0, maxv * 1.15)
    ax.tick_params(axis="y", length=0)

NOMES_CARGO = {
    "Analista de Dados/Data Analyst": "Analista de Dados",
    "Cientista de Dados/Data Scientist": "Cientista de Dados",
    "Engenheiro de Dados/Data Engineer/Data Architect": "Engenheiro de Dados",
    "Analista de BI/BI Analyst": "Analista de BI",
    "Analista de Negócios/Business Analyst": "Analista de Negócios",
    "Analytics Engineer": "Analytics Engineer",
    "Engenheiro de Machine Learning/ML Engineer/AI Engineer": "Eng. de ML / IA",
    "Desenvolvedor/ Engenheiro de Software/ Analista de Sistemas": "Dev / Eng. de Software",
    "Data Product Manager/ Product Manager (PM/APM/DPM/GPM/PO)": "Product Manager de Dados",
    "Arquiteto de Dados/Data Architect": "Arquiteto de Dados",
    "Analista de Suporte/Analista Técnico": "Analista de Suporte",
}


def g01_cargos(t, pasta):
    d = t["gold_cargos"].query("ano == 2025 and cargo != 'Outra Opção'").nlargest(8, "qtd")
    fig, ax = plt.subplots(figsize=(10, 5.6))
    cores = [AZUL if i < 3 else CINZA for i in range(len(d))]
    _barras_h(ax, [NOMES_CARGO.get(c, c) for c in d.cargo], d.pct_do_ano.tolist(), cores, _pct)
    return _salvar(fig, pasta, "g01_cargos_2025.png")


def g02_setores(silver_setor, pasta):
    d = silver_setor.query("setor != 'Outra Opção'").head(8)
    fig, ax = plt.subplots(figsize=(10, 5.6))
    cores = [LARANJA if "Finan" in s else CINZA for s in d.setor]
    _barras_h(ax, d.setor.tolist(), d.pct.tolist(), cores, _pct)
    return _salvar(fig, pasta, "g02_setores_2025.png")


def g03_salario_senioridade(t, pasta):
    d = t["gold_senioridade"].sort_values(["ordem", "ano"])
    niveis = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+", "Gestor"]
    fig, ax = plt.subplots(figsize=(11, 5.6))
    w = 0.26
    for i, ano in enumerate([2023, 2024, 2025]):
        sub = d[d.ano == ano].set_index("senioridade")
        xs, vs = [], []
        for j, n in enumerate(niveis):
            if n in sub.index:
                xs.append(j + (i - 1) * w); vs.append(sub.loc[n, "salario_medio"])
        ax.bar(xs, vs, width=w - 0.03, color=COR_ANO[ano], label=str(ano))
        if ano == 2025:
            for x, v in zip(xs, vs):
                ax.text(x, v + 300, _brl(v), ha="center", fontsize=12.5, color=TXT, fontweight=600)
    ax.set_xticks(range(len(niveis)))
    ax.set_xticklabels(["Júnior", "Pleno", "Sênior", "Especialista\n(só 2025)", "Gestor"], fontsize=14, color=TXT)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0)
    ax.legend(loc="upper left", ncol=3, fontsize=13)
    ax.set_ylim(0, 24000)
    return _salvar(fig, pasta, "g03_salario_senioridade.png")


def g04_salario_cargo(t, pasta):
    d = (t["gold_cargos"].query("ano == 2025 and qtd >= 30 and cargo != 'Outra Opção'")
         .nlargest(9, "salario_medio"))
    fig, ax = plt.subplots(figsize=(10, 5.8))
    cores = [AZUL if i < 2 else CINZA for i in range(len(d))]
    _barras_h(ax, [f"{NOMES_CARGO.get(c, c)}  ({p:.0f}% do mercado)"
                   for c, p in zip(d.cargo, d.pct_do_ano)],
              d.salario_medio.tolist(), cores, _brl)
    return _salvar(fig, pasta, "g04_salario_cargo_2025.png")


def g05_mulheres(t, pasta):
    g = t["gold_genero"].query("ano == 2025 and senioridade.notna()", engine="python")
    tot = g.groupby("senioridade").qtd.sum()
    fem = g[g.genero == "Feminino"].groupby("senioridade").qtd.sum()
    ordem = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+", "Gestor"]
    pct = (fem / tot * 100).reindex(ordem)
    fig, ax = plt.subplots(figsize=(10, 5.4))
    ax.bar(range(5), pct.values, color=LARANJA, width=0.58)
    for i, v in enumerate(pct.values):
        ax.text(i, v + 0.6, _pct(v), ha="center", fontsize=15, fontweight=600)
    ax.set_xticks(range(5)); ax.set_xticklabels(["Júnior", "Pleno", "Sênior", "Especialista", "Gestor"], fontsize=14, color=TXT)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0); ax.set_ylim(0, 34)
    return _salvar(fig, pasta, "g05_mulheres_por_nivel_2025.png")


def g06_gap_salarial(t, pasta):
    g = t["gold_genero"].query("ano == 2025 and senioridade.notna()", engine="python")
    p = g.pivot_table(index="senioridade", columns="genero", values="salario_medio")
    ordem = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+", "Gestor"]
    p = p.reindex(ordem)
    fig, ax = plt.subplots(figsize=(11, 5.6))
    w = 0.36
    x = range(len(ordem))
    ax.bar([i - w/2 for i in x], p["Masculino"], width=w - 0.03, color=AZUL, label="Homens")
    ax.bar([i + w/2 for i in x], p["Feminino"], width=w - 0.03, color=LARANJA, label="Mulheres")
    for i, n in enumerate(ordem):
        gap = (p.loc[n, "Masculino"] - p.loc[n, "Feminino"]) / p.loc[n, "Masculino"] * 100
        txt = f"−{gap:.0f}%" if gap > 1 else "≈ igual"
        ax.text(i, max(p.loc[n]) + 700, txt, ha="center", fontsize=14, fontweight=600,
                color=TXT if gap > 1 else TXT2)
    ax.set_xticks(list(x)); ax.set_xticklabels(["Júnior", "Pleno", "Sênior", "Especialista", "Gestor"], fontsize=14, color=TXT)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0); ax.set_ylim(0, 25000)
    ax.legend(loc="upper left", ncol=2, fontsize=13)
    return _salvar(fig, pasta, "g06_gap_salarial_2025.png")


def _linhas(ax, series, anos=(2023, 2024, 2025), ymax=100, ajuste_inicio=None):
    """series: lista de (rotulo, [v23, v24, v25], cor). Rótulo direto no fim.
    ajuste_inicio: deslocamento vertical do rótulo inicial, para evitar sobreposição."""
    ajuste_inicio = ajuste_inicio or [0] * len(series)
    for (rot, vals, cor), dy in zip(series, ajuste_inicio):
        ax.plot(anos, vals, color=cor, linewidth=3, marker="o", markersize=9,
                markeredgecolor="white", markeredgewidth=2)
        ax.text(anos[0] - 0.06, vals[0] + dy, _pct(vals[0]), ha="right", va="center", fontsize=13, color=TXT2)
        ax.text(anos[-1] + 0.06, vals[-1], f"{_pct(vals[-1])}  {rot}", ha="left", va="center",
                fontsize=14, color=TXT, fontweight=600)
    ax.set_xticks(anos); ax.set_xticklabels([str(a) for a in anos], fontsize=14, color=TXT)
    ax.set_xlim(anos[0] - 0.35, anos[-1] + 1.0); ax.set_ylim(0, ymax)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0)
    ax.spines["bottom"].set_color(GRADE)


def g07_cloud(t, pasta):
    d = t["gold_tecnologias"].query("categoria == 'Cloud'")
    s = lambda n: d[d.tecnologia == n].sort_values("ano").pct_uso.tolist()
    fig, ax = plt.subplots(figsize=(10, 5.4))
    _linhas(ax, [("AWS", s("AWS"), AZUL), ("Azure", s("Azure"), LARANJA), ("GCP", s("GCP"), VERDE_AGUA)], ymax=60,
           ajuste_inicio=[-2.2, 0, 2.2])
    return _salvar(fig, pasta, "g07_cloud_evolucao.png")


def g08_stack_2025(t, pasta):
    d = t["gold_tecnologias"].query("ano == 2025 and tecnologia in ['Python','SQL','Power BI','AWS','Azure','GCP','Looker','R','Tableau']")
    d = d.sort_values("pct_uso", ascending=False)
    cor = {"Linguagem": AZUL, "Cloud": LARANJA, "BI": VERDE_AGUA}
    fig, ax = plt.subplots(figsize=(10, 5.8))
    _barras_h(ax, d.tecnologia.tolist(), d.pct_uso.tolist(), [cor[c] for c in d.categoria], _pct, maxv=100)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=v, label=k) for k, v in cor.items()], loc="lower right", fontsize=13)
    return _salvar(fig, pasta, "g08_tecnologias_2025.png")


def g09_ia_uso(t, pasta):
    d = t["gold_ia_uso"].sort_values("ano")
    fig, ax = plt.subplots(figsize=(10, 5.4))
    _linhas(ax, [("usam IA generativa", d.pct_usa_ia.tolist(), AZUL),
                 ("empresa paga a ferramenta", d.pct_empresa_paga.tolist(), LARANJA),
                 ("usam só versão gratuita", d.pct_so_gratuita.tolist(), VERDE_AGUA)], ymax=105)
    ax.set_xlim(2022.65, 2026.6)
    return _salvar(fig, pasta, "g09_ia_uso.png")


def g10_ia_empresas(t, pasta):
    d = t["gold_ia_empresas"]
    prio = d[d.prioridade.isin(["Principal prioridade", "Entre as prioridades"])].groupby("ano").pct_do_ano.sum()
    nao = d[d.prioridade == "Não é prioridade"].set_index("ano").pct_do_ano
    fig, ax = plt.subplots(figsize=(10, 5.4))
    _linhas(ax, [("é prioridade na empresa", prio.sort_index().tolist(), AZUL),
                 ("não é prioridade", nao.sort_index().tolist(), CINZA)], ymax=75)
    return _salvar(fig, pasta, "g10_ia_prioridade_empresas.png")


def g11_regiao(t, pasta):
    d = t["gold_regiao"].query("ano == 2025").sort_values("qtd", ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    cores = [AZUL if r == "Sudeste" else CINZA for r in d.regiao]
    _barras_h(ax, [f"{r}  ·  {_pct(p)} dos profissionais" for r, p in zip(d.regiao, d.pct_do_ano)],
              d.salario_medio.tolist(), cores, _brl)
    return _salvar(fig, pasta, "g11_regiao_2025.png")


def g12_modelo(t, pasta):
    d = t["gold_modelo_trabalho"].query("ano == 2025")
    ordem = ["Remoto", "Híbrido flexível", "Híbrido fixo", "Presencial"]
    at = d[d.tipo == "Atual"].set_index("modelo").pct_do_ano.reindex(ordem)
    idl = d[d.tipo == "Ideal"].set_index("modelo").pct_do_ano.reindex(ordem)
    fig, ax = plt.subplots(figsize=(11, 5.4))
    w = 0.36
    x = range(4)
    ax.bar([i - w/2 for i in x], at, width=w - 0.03, color=CINZA, label="Como trabalha hoje")
    ax.bar([i + w/2 for i in x], idl, width=w - 0.03, color=AZUL, label="Como gostaria de trabalhar")
    for i in x:
        ax.text(i - w/2, at.iloc[i] + 0.8, _pct(at.iloc[i]), ha="center", fontsize=13, color=TXT2)
        ax.text(i + w/2, idl.iloc[i] + 0.8, _pct(idl.iloc[i]), ha="center", fontsize=14, fontweight=600)
    ax.set_xticks(list(x)); ax.set_xticklabels(ordem, fontsize=14, color=TXT)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0); ax.set_ylim(0, 52)
    ax.legend(loc="upper right", fontsize=13)
    return _salvar(fig, pasta, "g12_modelo_trabalho_2025.png")


def g13_satisfacao(satisf, pasta):
    d = satisf.set_index("modelo_trabalho_atual").reindex(["Híbrido flexível", "Remoto", "Híbrido fixo", "Presencial"])
    fig, ax = plt.subplots(figsize=(10, 4.6))
    cores = [LARANJA if m == "Presencial" else CINZA for m in d.index]
    _barras_h(ax, d.index.tolist(), d.pct_satisfeitos.tolist(), cores, _pct, maxv=100)
    return _salvar(fig, pasta, "g13_satisfacao_modelo_2025.png")


def gerar_todos(t, extras, pasta="graficos"):
    """t: tabelas gold; extras: {'setores': df, 'satisfacao': df} vindos da Silver."""
    os.makedirs(pasta, exist_ok=True)
    return [
        g01_cargos(t, pasta), g02_setores(extras["setores"], pasta),
        g03_salario_senioridade(t, pasta), g04_salario_cargo(t, pasta),
        g05_mulheres(t, pasta), g06_gap_salarial(t, pasta),
        g07_cloud(t, pasta), g08_stack_2025(t, pasta),
        g09_ia_uso(t, pasta), g10_ia_empresas(t, pasta),
        g11_regiao(t, pasta), g12_modelo(t, pasta), g13_satisfacao(extras["satisfacao"], pasta),
    ]


In [ ]:
tabelas = {n: spark.read.parquet(f"{GOLD}/{n}").toPandas() for n in [
    "gold_kpis", "gold_cargos", "gold_senioridade", "gold_genero", "gold_tecnologias",
    "gold_ia_uso", "gold_ia_empresas", "gold_regiao", "gold_modelo_trabalho"]}

silver = spark.read.parquet(SILVER)
silver.createOrReplaceTempView("silver_survey")
setores = spark.sql("""
    SELECT setor, COUNT(*) AS qtd, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM silver_survey WHERE ano = 2025 AND setor IS NOT NULL
    GROUP BY setor ORDER BY qtd DESC""").toPandas()
satisfacao = spark.sql("""
    SELECT modelo_trabalho_atual, COUNT(*) AS qtd, ROUND(AVG(satisfeito) * 100, 1) AS pct_satisfeitos
    FROM silver_survey WHERE ano = 2025 AND modelo_trabalho_atual IS NOT NULL AND satisfeito IS NOT NULL
    GROUP BY modelo_trabalho_atual""").toPandas()

arquivos = gerar_todos(tabelas, {"setores": setores, "satisfacao": satisfacao}, pasta="/tmp/graficos")

# envia os PNGs para o S3, junto com o restante do projeto
for f in arquivos:
    s3.upload_file(f, BUCKET, "graficos/" + f.split("/")[-1])
print(len(arquivos), "gráficos gerados")

## 7. Limitações e cuidados na leitura

1. **Salário é estimado** pelo ponto médio da faixa (não é o valor exato); a mediana fica “arredondada” por isso.
2. A pesquisa é **voluntária** (comunidade Data Hackers): retrata quem responde, não todo o mercado. 2025 tem menos respostas (3.494).
3. **“Especialista/Staff+”** só existe em 2025; comparações de senioridade ao longo do tempo usam Júnior, Pleno, Sênior e Gestor.
4. A pergunta de prioridade de IA nas empresas é respondida **só por gestores**; a de uso de IA, **só por não gestores**.
5. Relações como “quem tem IA paga pela empresa ganha mais” são **correlações**, não causa e efeito.
6. Norte e Centro-Oeste têm amostras pequenas (48 e 231 respostas em 2025): diferenças de salário nessas regiões devem ser lidas com cautela.